# data_analysis.py file Test with Data from Intel Device

In [ ]:
import data_analysis as da
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "N:\\W26-S26 Baugh Lab Coop\\Measurement Data 3D1S FC\\"

### Turn-on

In [ ]:
file_path_turnon = file_path + "Turnon.csv"
df = pd.read_csv(file_path_turnon)
Xdata = df["P0 (V)"].to_numpy()
TDdata = df["Triple Dot Voltage (V)"].to_numpy()
SETData = df["SET Voltage (V)"].to_numpy()

print("Triple Dot Turn on:")
TD_turnon = da.extract_turn_on_voltage(Xdata, TDdata)

# print("SET Turn on:")
# SET_turnon = da.extract_turn_on_voltage(Xdata, SETData)

### Pinch-Off Curves First Cooldown

In [ ]:
files = ["B0PO.csv", "B1PO.csv", "B2PO.csv", "B3PO.csv", "P0PO.csv", "P1PO.csv", "P2PO.csv"]
for file in files:
    file_path_po = file_path + file
    df = pd.read_csv(file_path_po)
    name = df.columns[1]
    Xdata = df[name].to_numpy()
    TDdata = df["Triple Dot Voltage (V)"].to_numpy()*1e-7*1e9 #nA

    print("Gate pinch-off:")
    voltage_window = da.pinch_off_curve_ranges(Xdata, TDdata)

files = ["B20PO.csv", "B21PO.csv", "P20PO.csv"]
for file in files:
    file_path_po = file_path + file
    df = pd.read_csv(file_path_po)
    name = df.columns[1]
    Xdata = df[name].to_numpy()
    SETdata = df["SET Voltage (V)"].to_numpy()*1e-8*1e9 #nA

    print("Gate pinch-off:")
    voltage_window = da.pinch_off_curve_ranges(Xdata, SETdata)

### Resweeping Barrier Gates

In [ ]:
files = ["B20S_2.csv", "B21S_2.csv"]
barrier_pinch_offs = []
for file in files:
    file_path_bg = file_path + file
    df = pd.read_csv(file_path_bg)
    name = df.columns[1]
    Xdata = df[name].to_numpy()
    TDdata = df["SET Voltage (V)"].to_numpy()*1e-7*1e9 #nA
    nf = np.average(TDdata[-20:])

    print("Barrier gate Resweep:")
    voltage_window, pinch_off_fig = da.pinch_off_curve_ranges(Xdata, TDdata, nf)
    barrier_pinch_offs.append(voltage_window[0])
    display(pinch_off_fig)
    
print("Barrier pinch-off voltages:", barrier_pinch_offs)

### Pinch-off for Second Cooldown

In [ ]:
files = ["AC0PO.csv", "AC1PO.csv", "B0PO_2.csv", "B1PO_2.csv", "B2PO_2.csv", "B3PO_2.csv", "P0PO_2.csv", "P1PO_2.csv", "P2PO_2.csv"]
for file in files:
    file_path_po = file_path + file
    df = pd.read_csv(file_path_po)
    name = df.columns[1]
    Xdata = df[name].to_numpy()
    TDdata = df["Triple Dot Signal (V)"].to_numpy()*1e-8*1e9 #nA

    print("Gate pinch-off:")
    voltage_window = da.pinch_off_curve_ranges(Xdata, TDdata)

files = ["AC2PO.csv", "AC3PO.csv", "B20PO_2.csv", "B21PO_2.csv", "P20PO_2.csv"]
for file in files:
    file_path_po = file_path + file
    df = pd.read_csv(file_path_po)
    name = df.columns[1]
    Xdata = df[name].to_numpy()
    SETdata = df["SET Signal (V)"].to_numpy()*1e-8*1e9 #nA

    print("Gate pinch-off:")
    voltage_window = da.pinch_off_curve_ranges(Xdata, SETdata)

### Barrier-Barrier Scan for First Cooldown

In [ ]:
file_path_bbs = file_path + "Left Sensor Barrier Gate_Right Sensor Barrier Gate_Scan_2026-06-12_08-22-11.csv"
df = pd.read_csv(file_path_bbs)
lbdata = df["spi_rack.module1.dac8.voltage"].to_numpy()
rbdata = df["spi_rack.module1.dac7.voltage"].to_numpy()
TDdata = df["agilent_left.volt"].to_numpy()*1e-8*1e9 #nA
SETdata = df["agilent_right.volt"].to_numpy()*1e-8*1e9 #nA

print("B20-B21 Sweep:")
for i in range(1):
    best_point, working_points, perp_traces, fig = da.extract_working_point(lbdata, rbdata, SETdata, ["B20", "B21"], 'SET', barrier_pinch_offs=(0.85, 0.85), debug=False, plot_results=True)
    display(fig)
    print(best_point)

### Coulomb Oscillations: Finding max conductance pts

In [ ]:
file_path_co = file_path + "P20S.csv"
df = pd.read_csv(file_path_co)
Xdata = df["P20 (V)"].to_numpy()
SETData = df["SET Voltage (V)"].to_numpy()*1e-7*1e9 #nA

best_pts, _ = da.extract_max_conductance_points(Xdata, SETData)
print(best_pts)

### Dot - Lead Tuning

In [ ]:
file_path_dl = file_path + "P0B0S.csv" # P2B3S
df = pd.read_csv(file_path_dl)
Xdata = df["P0 (V)"].to_numpy()
Ydata = df["B0 (V)"].to_numpy()
SETData = df["SET Voltage (V)"].to_numpy()*1e-7*1e9 #nA

best_sens_pts_list, all_sens_pts_list = da.extract_tunnel_barrier_latching(Xdata, Ydata, SETData)
print(all_sens_pts_list)